# CO-SPY Fusion v3 từ checkpoint có sẵn

Notebook này chỉ train Fusion v3; không train lại Semantic, Artifact hoặc NPR. Thêm `Co-Spy_v1` và `Co-Spy_v2` vào Kaggle, bật GPU và Internet, rồi Run All.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO = Path('/kaggle/working/CO-SPY')
if not (REPO / '.git').exists():
    subprocess.run(['git','clone','https://github.com/datnguyentrung/CO-SPY.git',str(REPO)], check=True)
os.chdir(REPO)
print('Repository:', REPO)
print('Commit:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())

# Tự áp dụng patch nếu GitHub còn commit cũ.
fusion_file = REPO / 'detectors' / 'sd_v1_4' / 'fusion.py'
fusion = fusion_file.read_text(encoding='utf-8')
fusion = fusion.replace('feature_dims = [self.sem.fc.in_features, self.art.fc.in_features]', 'feature_dims = [self.sem.fc.in_features + self.sem.fc.out_features, self.art.fc.in_features]')
fusion = fusion.replace('sem_feat = self.sem.clip.encode_image(x_sem)', '            sem_embedding = self.sem.clip.encode_image(x_sem)\n            sem_logit = self.sem.fc(sem_embedding)\n            sem_feat = torch.cat([sem_embedding, sem_logit], dim=1)')
fusion_file.write_text(fusion, encoding='utf-8')
train_file = REPO / 'train.py'
train = train_file.read_text(encoding='utf-8')
if 'semantic_feature_schema=clip_plus_trained_logit_v1' not in train:
    train = train.replace('            self.cache_key = hashlib.sha256(', '            if self.fusion_head != \"v1\":\n                signature += \"|semantic_feature_schema=clip_plus_trained_logit_v1\"\n            self.cache_key = hashlib.sha256(')
    train_file.write_text(train, encoding='utf-8')
assert 'sem_logit = self.sem.fc(sem_embedding)' in fusion_file.read_text(encoding='utf-8')
print('Fusion v3 patch: OK')

In [ ]:
packages=['diffusers==0.32.2','open-clip-torch==2.31.0','timm==1.0.15','loguru==0.7.3','accelerate==1.0.1','ftfy==6.2.3','transformers==4.46.3','safetensors==0.5.3']
subprocess.run([sys.executable,'-m','pip','install','-q',*packages], check=True)
print('Dependencies installed')

In [ ]:
INPUT=Path('/kaggle/input')
def find_dir(name):
    hits=[p for p in INPUT.rglob(name) if p.is_dir()]
    if not hits: raise FileNotFoundError(f'Không tìm thấy {name} trong /kaggle/input')
    return hits[0]
dataset_candidates=[p for p in INPUT.rglob('*') if p.is_dir() and (p/'mscoco2017').is_dir() and (p/'stable-diffusion-v1-4').is_dir()]
if not dataset_candidates: raise FileNotFoundError('Không tìm thấy Co-Spy_v1')
data_root=dataset_candidates[0]
ckpt_root=find_dir('ckpt')
print('Dataset root:',data_root)
print('Checkpoint root:',ckpt_root)
data_link=REPO/'data'/'train'/'sd-v1_4'
data_link.mkdir(parents=True,exist_ok=True)
for name in ['mscoco2017','stable-diffusion-v1-4']:
    link=data_link/name
    source=data_root/name
    nested=source/name
    if nested.is_dir(): source=nested
    if link.is_symlink(): link.unlink()
    elif link.exists(): shutil.rmtree(link)
    link.symlink_to(source,target_is_directory=True)
    print(name,'->',source)
    for split in ['train2017','val2017']:
        if not (link/split).is_dir(): raise RuntimeError(f'Thiếu {link/split}')
target_ckpt=REPO/'ckpt'
if target_ckpt.exists(): shutil.rmtree(target_ckpt)
shutil.copytree(ckpt_root,target_ckpt)
for branch in ['semantic','artifact','npr']:
    files=list((target_ckpt/'sd-v1_4'/branch).rglob('checkpoint_best.pth'))
    if not files: raise FileNotFoundError(f'Thiếu checkpoint {branch}')
    print(branch,files[0],files[0].stat().st_size/1024**2,'MB')

In [ ]:
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(),'GPUs:',torch.cuda.device_count())
if not torch.cuda.is_available(): raise RuntimeError('Hãy bật Accelerator = GPU')
!python -m py_compile detectors/sd_v1_4/fusion.py train.py

In [ ]:
!python -u main.py --phase train --mode fusion --train_dataset sd-v1_4 --ckpt /kaggle/working/CO-SPY/ckpt --fusion_head v3 --epochs 5 --early-stopping-patience 2 --artifact_views 1 --num-workers 2 --prefetch-factor 2 --precision auto

In [ ]:
for p in sorted((REPO/'ckpt'/'sd-v1_4'/'fusion_v3').rglob('*')):
    if p.is_file(): print(p,f'{p.stat().st_size/1024**2:.2f} MB')
print('Hoàn tất. Hãy Save Version để giữ checkpoint và feature-cache.')